Notebook, um die Eventannotationen zu aufzubereiten und extrahieren, um mit crowdkit probabilistische Goldstandards zu erstellen.

Was passiert hier?
- Manche Annotationsitems enthalten leading whitespaces, diese werden entfernt und der Startindex entsprechend angepasst
- Alle vorhandenen Annotationsspannen werden pro Text und AnnotatorIn extrahiert
- Teilweise sind Annotationsspannen diskontinuierlich, die werden reduziert, wobei der aller erste Span und der aller letzte Span zusammengeführt werden als bounding spans
- Pro Text werden die Spannen der Annotierenden (2) + Goldstandard für spätere Vergleiche gematcht. Folgende Matches werden extrahiert:
    - Alle 3 Annotationskollektionen haben den gleichen Start- und Endindex und werden anschließend für weitere Matches exkludiert
    - 2 Annotationskollektionen haben den gleichen Start- und Endindex und 1 weitere teilt den gleichen Startindex, auch diese werden in der nächsten Runde zum Matchen exkludiert
    - Für alle übrigen Spannen wird geschaut, ob sie den gleichen Startindex haben

- Auf die gematchten Spans werden IDs gemappt.
- Basierend auf den Matches werden dann die relevanten Annotationen extrahiert. Diese werden in "261112_item_mapped_final_data.json" gespeichert, wobei pro ID (key), die Annotionen aufgelöst werden

- Der finale Dataframe wird vorbereitet mit den Columns: title, item, task, worker und label (Vorgabe crowd-kit) und in "261211_final_data.csv" gespeichert

- Am Ende des Notebooks noch einige Möglichkeiten, um die Mappings zu prüfen:
    - Haben gleiche Textsegmente unterschiedliche Task-IDs?
    - Liegen für die selbe Task-ID unterscheidliche Textsegmente vor?

In [ ]:
import json
import pandas as pd

## Read Event annotations

In [256]:
with open ("../data/Annotations_EvENT.json") as file:
    eventannodata = json.load(file)

## Get simple numbers

In [257]:
for title, annotations in eventannodata.items():
    print(title)
    for annotator, annotionen in annotations.items():
        print(annotator, "mit N annotaitonen: ", len(annotionen))

Das Erdbeben in Chili
Annotator1 mit N annotaitonen:  710
Annotator2 mit N annotaitonen:  715
gold_standard mit N annotaitonen:  712
Der blonde Eckbert
Annotator1 mit N annotaitonen:  924
Annotator2 mit N annotaitonen:  937
gold_standard mit N annotaitonen:  930
Die Judenbuche
Annotator1 mit N annotaitonen:  2476
Annotator2 mit N annotaitonen:  2383
gold_standard mit N annotaitonen:  2491
Die Verwandlung
Annotator1 mit N annotaitonen:  2352
Annotator2 mit N annotaitonen:  2442
gold_standard mit N annotaitonen:  2364
Effi Briest
Annotator1 mit N annotaitonen:  6884
Annotator2 mit N annotaitonen:  6745
gold_standard mit N annotaitonen:  6666
Krambambuli
Annotator1 mit N annotaitonen:  565
Annotator2 mit N annotaitonen:  575
gold_standard mit N annotaitonen:  571


# Clean Data (remove starting white spaces)

- Iterate over annotated text: for each annotated text, if it startswith space --> delete and correct span

In [258]:
print(len(eventannodata))

# Copy data
copy_eventdata = eventannodata.copy()
print(len(copy_eventdata))

6
6


In [259]:

#key is title and contains a dict of annotations, where each annotator has his/her own dict again
for title,annotations in copy_eventdata.items():
      for annotator, annotationen in annotations.items():

         for listofannotations in annotationen:
            # get each item and tag and put them in one dict
            
            current_anno_item = listofannotations.get("annotation")
            #print("Current anno item: ",current_anno_item, " with length: ",len(current_anno_item))
            if current_anno_item.startswith(" "):
                #print(len(current_anno_item))
                
                #Führende Leerzeichen entfernen
                current_anno_item_stripped = current_anno_item.lstrip(" ")
                # print("mutated: ", current_anno_item_stripped, " with length: ",len(current_anno_item_stripped))

                #adapt span start, wenn ein whitespace entfernt wurde
                for span in listofannotations.get("spans"):
                    # print(span)
                    span[0] += 1
                    # print(span)
            
                listofannotations["annotation"] = current_anno_item_stripped
            



# 2. New approach: (1) Extrahiere pro Text und AnnotatorIn alle Annotationsspannen

In [249]:
#key is title and contains a dict of annotations, where each annotator has his/her own dict again

mapping_title_annospans = dict() #dict für alle titel


for title,annotations in copy_eventdata.items():
    print(title)

    mapping_annotator_spans = dict() #dict für alle annotatorInnen eines titels
    for annotator, annotationen in annotations.items():
        # list of anno spans for each annotator
        annotator_spans = list()

        for listofannotations in annotationen:
            annotator_spans.append(listofannotations.get("spans"))

        mapping_annotator_spans[annotator] = annotator_spans

        print(annotator, ": ",len(annotator_spans))
        #print(annotator_spans)

    mapping_title_annospans[title] = mapping_annotator_spans

Das Erdbeben in Chili
Annotator1 :  710
Annotator2 :  715
gold_standard :  712
Der blonde Eckbert
Annotator1 :  924
Annotator2 :  937
gold_standard :  930
Die Judenbuche
Annotator1 :  2476
Annotator2 :  2383
gold_standard :  2491
Die Verwandlung
Annotator1 :  2352
Annotator2 :  2442
gold_standard :  2364
Effi Briest
Annotator1 :  6884
Annotator2 :  6745
gold_standard :  6666
Krambambuli
Annotator1 :  565
Annotator2 :  575
gold_standard :  571


#### Spans die länger als 1 sind

In [225]:
# for k,v in mapping_title_annospans.items():
#     print(k, v)
#     for anno, spans in v.items():
#         for span in spans:
#             if len(span)>1:
#                 print(span)
#         print(spans)

## 2.2. Bounding Spans erstellen, d.h. diskontinuierliche Spans zu einem Span mappen

In [226]:
from itertools import combinations

bounding_spans = dict()

for title, annotators in mapping_title_annospans.items():

    bounding_spans[title] = {}

    for annotator, span_lists in annotators.items():

        spans = []

        for span_list in span_lists:

            if len(span_list) == 1:
                spans.append(tuple(span_list[0]))

            elif len(span_list) > 1:
                merged_span = (span_list[0][0], span_list[-1][1])
                spans.append(merged_span)

        bounding_spans[title][annotator] = spans

## 2. Matchen
- wo start und index exact gematcht wird in allen 3 kollektionen
- wo exakt auf start und ende in 2 gematcht wird und in einer zusätzlichen nur der start
- gleicher start bei allen drei, aber überall unterschiedliche Endindizes

In [227]:
matching_results = dict()

for title, collections in bounding_spans.items():

    names = list(collections.keys())

    # Jede Annotation bekommt eine eindeutige ID
    remaining = {
        name: list(enumerate(collections[name]))
        for name in names
    }

    results = {
        "exact_3": [],
        "exact_2_start_1": [],
        "start_3": [],
        # "exact_2": [],
        # "start_2": []
    }

    # Hilfsfunktion: Annotation aus Collection entfernen
    def consume(match):
        for name, idx, span in match:
            remaining[name] = [
                (i, s) for i, s in remaining[name] if i != idx
            ]

    # -----------------------------------
    # STUFE 1: Exact Match in allen 3
    # -----------------------------------

    common = set.intersection(*[
        {span for idx, span in remaining[name]}
        for name in names
    ])

    for span in common:

        match = []

        for name in names:
            idx = next(
                i for i, s in remaining[name] if s == span
            )
            match.append((name, idx, span))

        results["exact_3"].append(match)
        consume(match)

    # -----------------------------------
    # STUFE 2: Exact Match zwischen 2
    #         + Start Match in dritter
    # -----------------------------------

    for pair in combinations(names, 2):

        third = [n for n in names if n not in pair][0]

        common = (
            {s for i, s in remaining[pair[0]]}
            &
            {s for i, s in remaining[pair[1]]}
        )

        for span in common:

            idx_a = next(i for i, s in remaining[pair[0]] if s == span)
            idx_b = next(i for i, s in remaining[pair[1]] if s == span)

            # In dritter Collection nach gleichem Start suchen
            third_matches = [
                (i, s) for i, s in remaining[third]
                if s[0] == span[0]
            ]

            if third_matches:

                idx_c, span_c = third_matches[0]

                match = [
                    (pair[0], idx_a, span),
                    (pair[1], idx_b, span),
                    (third, idx_c, span_c)
                ]

                results["exact_2_start_1"].append(match)
                consume(match)

            # else:

            #     match = [
            #         (pair[0], idx_a, span),
            #         (pair[1], idx_b, span)
            #     ]

            #     results["exact_2"].append(match)
            #     consume(match)

    # -----------------------------------
    # STUFE 3: Start Match in allen 3
    # -----------------------------------

    common_starts = set.intersection(*[
        {s[0] for i, s in remaining[name]}
        for name in names
    ])

    for start in common_starts:

        match = []

        for name in names:
            idx, span = next(
                (i, s) for i, s in remaining[name]
                if s[0] == start
            )

            match.append((name, idx, span))

        results["start_3"].append(match)
        consume(match)

    # # -----------------------------------
    # # STUFE 4: Start Match zwischen 2
    # # -----------------------------------

    # for pair in combinations(names, 2):

    #     common_starts = (
    #         {s[0] for i, s in remaining[pair[0]]}
    #         &
    #         {s[0] for i, s in remaining[pair[1]]}
    #     )

    #     for start in common_starts:

    #         idx_a, span_a = next(
    #             (i, s) for i, s in remaining[pair[0]]
    #             if s[0] == start
    #         )

    #         idx_b, span_b = next(
    #             (i, s) for i, s in remaining[pair[1]]
    #             if s[0] == start
    #         )

    #         match = [
    #             (pair[0], idx_a, span_a),
    #             (pair[1], idx_b, span_b)
    #         ]

    #         results["start_2"].append(match)
    #         consume(match)

    matching_results[title] = {
        "matches": results,
        "unmatched": remaining
    }

### Kurz checken

In [228]:
for title, data in matching_results.items():

    print("\n", title)

    for match_type, matches in data["matches"].items():
        print(match_type, len(matches))
        if match_type == "exact_2_start_1":
            print(matches)

    print("Unmatched:")

    for annotator, spans in data["unmatched"].items():
        print(annotator, len(spans))


 Das Erdbeben in Chili
exact_3 678
exact_2_start_1 18
[[('Annotator1', 363, (19060, 19409)), ('gold_standard', 363, (19060, 19409)), ('Annotator2', 365, (19060, 19288))], [('Annotator1', 343, (17949, 18072)), ('gold_standard', 343, (17949, 18072)), ('Annotator2', 345, (17949, 17973))], [('Annotator1', 293, (15331, 15379)), ('gold_standard', 293, (15331, 15379)), ('Annotator2', 295, (15331, 15378))], [('Annotator1', 665, (32760, 32830)), ('gold_standard', 667, (32760, 32830)), ('Annotator2', 672, (32760, 32829))], [('Annotator1', 112, (6540, 6645)), ('gold_standard', 112, (6540, 6645)), ('Annotator2', 112, (6540, 6562))], [('Annotator1', 623, (31263, 31274)), ('gold_standard', 625, (31263, 31274)), ('Annotator2', 630, (31263, 31322))], [('Annotator1', 558, (29103, 29176)), ('gold_standard', 560, (29103, 29176)), ('Annotator2', 563, (29103, 29143))], [('Annotator1', 520, (27656, 27676)), ('gold_standard', 522, (27656, 27676)), ('Annotator2', 523, (27656, 27669))], [('Annotator1', 131, (

## Extract matching spans regardless of the type into one list per title

In [229]:
title_matches = dict()
for title, data in matching_results.items():
    matches_flat_list = list()
    for match_type, matches in data["matches"].items():
        
        for match in matches:
            matches_flat_list.append(match)
            

    title_matches[title] = list(matches_flat_list)

In [230]:
for title, matches in title_matches.items():
    print(title, ": ", len(matches))
    print(matches)

Das Erdbeben in Chili :  696
[[('Annotator1', 20, (1563, 1643)), ('Annotator2', 20, (1563, 1643)), ('gold_standard', 20, (1563, 1643))], [('Annotator1', 393, (21023, 21041)), ('Annotator2', 397, (21023, 21041)), ('gold_standard', 393, (21023, 21041))], [('Annotator1', 36, (2590, 2687)), ('Annotator2', 37, (2590, 2687)), ('gold_standard', 36, (2590, 2687))], [('Annotator1', 183, (9916, 9965)), ('Annotator2', 184, (9916, 9965)), ('gold_standard', 183, (9916, 9965))], [('Annotator1', 569, (29537, 29555)), ('Annotator2', 575, (29537, 29555)), ('gold_standard', 571, (29537, 29555))], [('Annotator1', 695, (34269, 34384)), ('Annotator2', 701, (34269, 34384)), ('gold_standard', 697, (34269, 34384))], [('Annotator1', 160, (8786, 8834)), ('Annotator2', 161, (8786, 8834)), ('gold_standard', 160, (8786, 8834))], [('Annotator1', 201, (11005, 11060)), ('Annotator2', 202, (11005, 11060)), ('gold_standard', 201, (11005, 11060))], [('Annotator1', 570, (29557, 29560)), ('Annotator2', 576, (29557, 29560)

## Now Map IDs to the matches

In [242]:
match_mapping = {}

for title, data in matching_results.items():

    match_mapping[title] = {}
    match_id = 0

    for match_type, matches in data["matches"].items():

        for match in matches:
            match_id += 1

            for annotator, index, span in match:
                match_mapping[title][(annotator, index, span)] = match_id

In [243]:
for k,v in match_mapping.items():
    print(v)

{('Annotator1', 20, (1563, 1643)): 1, ('Annotator2', 20, (1563, 1643)): 1, ('gold_standard', 20, (1563, 1643)): 1, ('Annotator1', 393, (21023, 21041)): 2, ('Annotator2', 397, (21023, 21041)): 2, ('gold_standard', 393, (21023, 21041)): 2, ('Annotator1', 36, (2590, 2687)): 3, ('Annotator2', 37, (2590, 2687)): 3, ('gold_standard', 36, (2590, 2687)): 3, ('Annotator1', 183, (9916, 9965)): 4, ('Annotator2', 184, (9916, 9965)): 4, ('gold_standard', 183, (9916, 9965)): 4, ('Annotator1', 569, (29537, 29555)): 5, ('Annotator2', 575, (29537, 29555)): 5, ('gold_standard', 571, (29537, 29555)): 5, ('Annotator1', 695, (34269, 34384)): 6, ('Annotator2', 701, (34269, 34384)): 6, ('gold_standard', 697, (34269, 34384)): 6, ('Annotator1', 160, (8786, 8834)): 7, ('Annotator2', 161, (8786, 8834)): 7, ('gold_standard', 160, (8786, 8834)): 7, ('Annotator1', 201, (11005, 11060)): 8, ('Annotator2', 202, (11005, 11060)): 8, ('gold_standard', 201, (11005, 11060)): 8, ('Annotator1', 570, (29557, 29560)): 9, ('Ann

## Jetzt spans im og dataframe collapsen (boundary spans)

In [260]:

#key is title and contains a dict of annotations, where each annotator has his/her own dict again
for title,annotations in copy_eventdata.items():
      for annotator, annotationen in annotations.items():

         for listofannotations in annotationen:
            # get each item and tag and put them in one dict
            
            currentspan = listofannotations.get("spans")
            #print("Current anno item: ",current_anno_item, " with length: ",len(current_anno_item))
          
            for spans in currentspan:

                if len(currentspan) >1:
                    print("before: ",currentspan)
                    merged_span = (currentspan[0][0], currentspan[-1][1])
        
                    print("after: ",merged_span)
                    # print(span)
            
                    listofannotations["spans"] = merged_span

                else:
                    listofannotations["spans"] = tuple(currentspan)
                    
            



before:  [[22, 147], [208, 318]]
after:  (22, 318)
before:  [[22, 147], [208, 318]]
after:  (22, 318)
before:  [[384, 497], [532, 540]]
after:  (384, 540)
before:  [[384, 497], [532, 540]]
after:  (384, 540)
before:  [[651, 674], [824, 849]]
after:  (651, 849)
before:  [[651, 674], [824, 849]]
after:  (651, 849)
before:  [[676, 693], [747, 822]]
after:  (676, 822)
before:  [[676, 693], [747, 822]]
after:  (676, 822)
before:  [[1166, 1206], [1237, 1259]]
after:  (1166, 1259)
before:  [[1166, 1206], [1237, 1259]]
after:  (1166, 1259)
before:  [[1818, 1908], [1997, 2008], [2060, 2074]]
after:  (1818, 2074)
before:  [[1818, 1908], [1997, 2008], [2060, 2074]]
after:  (1818, 2074)
before:  [[1818, 1908], [1997, 2008], [2060, 2074]]
after:  (1818, 2074)
before:  [[2076, 2081], [2105, 2108]]
after:  (2076, 2108)
before:  [[2076, 2081], [2105, 2108]]
after:  (2076, 2108)
before:  [[2110, 2126], [2157, 2296]]
after:  (2110, 2296)
before:  [[2110, 2126], [2157, 2296]]
after:  (2110, 2296)
before:

In [ ]:

# ## Save collapsed and cleaned event data to json
# with open("../data/261112_cleaned_collapsed_eventdata.json", "w") as f:
#    json.dump(copy_eventdata, f, indent=4)

## Finales dictionary erstellen: Now iterate over the data and extract item, tag for each span in match_mapping

In [263]:
for k,v in match_mapping.items():
    print(v)

{('Annotator1', 20, (1563, 1643)): 1, ('Annotator2', 20, (1563, 1643)): 1, ('gold_standard', 20, (1563, 1643)): 1, ('Annotator1', 393, (21023, 21041)): 2, ('Annotator2', 397, (21023, 21041)): 2, ('gold_standard', 393, (21023, 21041)): 2, ('Annotator1', 36, (2590, 2687)): 3, ('Annotator2', 37, (2590, 2687)): 3, ('gold_standard', 36, (2590, 2687)): 3, ('Annotator1', 183, (9916, 9965)): 4, ('Annotator2', 184, (9916, 9965)): 4, ('gold_standard', 183, (9916, 9965)): 4, ('Annotator1', 569, (29537, 29555)): 5, ('Annotator2', 575, (29537, 29555)): 5, ('gold_standard', 571, (29537, 29555)): 5, ('Annotator1', 695, (34269, 34384)): 6, ('Annotator2', 701, (34269, 34384)): 6, ('gold_standard', 697, (34269, 34384)): 6, ('Annotator1', 160, (8786, 8834)): 7, ('Annotator2', 161, (8786, 8834)): 7, ('gold_standard', 160, (8786, 8834)): 7, ('Annotator1', 201, (11005, 11060)): 8, ('Annotator2', 202, (11005, 11060)): 8, ('gold_standard', 201, (11005, 11060)): 8, ('Annotator1', 570, (29557, 29560)): 9, ('Ann

In [269]:
final_dict = dict() 

for title, matchedannos in match_mapping.items():

    id_anno_map = dict()
    for annotuple,match_id in matchedannos.items():

        annotator = annotuple[0]
        index = annotuple[1]

        annotation = copy_eventdata[title][annotator][index]

        text = annotation["annotation"]
        spans = annotation["spans"]
        tag = annotation["tag"]

        print(match_id, annotator, tag, spans, annotuple[2])

        #eintrag für jede ID
        if match_id not in id_anno_map:
            id_anno_map[match_id] = {
                "annotations": {}
            }

        id_anno_map[match_id]["annotations"][annotator] = {
            
            "span": spans,
            "text": text,
            "tag": tag,
        }


    final_dict[title] = id_anno_map


1 Annotator1 process ([1563, 1643],) (1563, 1643)
1 Annotator2 process ([1563, 1643],) (1563, 1643)
1 gold_standard process ([1563, 1643],) (1563, 1643)
2 Annotator1 process ([21023, 21041],) (21023, 21041)
2 Annotator2 process ([21023, 21041],) (21023, 21041)
2 gold_standard process ([21023, 21041],) (21023, 21041)
3 Annotator1 stative_event (2590, 2687) (2590, 2687)
3 Annotator2 stative_event (2590, 2687) (2590, 2687)
3 gold_standard stative_event (2590, 2687) (2590, 2687)
4 Annotator1 process ([9916, 9965],) (9916, 9965)
4 Annotator2 process ([9916, 9965],) (9916, 9965)
4 gold_standard process ([9916, 9965],) (9916, 9965)
5 Annotator1 process ([29537, 29555],) (29537, 29555)
5 Annotator2 process ([29537, 29555],) (29537, 29555)
5 gold_standard process ([29537, 29555],) (29537, 29555)
6 Annotator1 non_event ([34269, 34384],) (34269, 34384)
6 Annotator2 non_event ([34269, 34384],) (34269, 34384)
6 gold_standard non_event ([34269, 34384],) (34269, 34384)
7 Annotator1 stative_event ([87

In [285]:
for k,v in final_dict.items():
    print(v)

{1: {'annotations': {'Annotator1': {'span': ([1563, 1643],), 'text': 'als ihr  schon, auf Befehl des Erzbischofs, der geschärfteste Prozeß  gemacht ward', 'tag': 'process'}, 'Annotator2': {'span': ([1563, 1643],), 'text': 'als ihr schon, auf Befehl des Erzbischofs, der geschärfteste Prozeß  gemacht ward', 'tag': 'process'}, 'gold_standard': {'span': ([1563, 1643],), 'text': 'als ihr  schon, auf Befehl des Erzbischofs, der geschärfteste Prozeß  gemacht ward', 'tag': 'process'}}}, 2: {'annotations': {'Annotator1': {'span': ([21023, 21041],), 'text': 'Josephe antwortete', 'tag': 'process'}, 'Annotator2': {'span': ([21023, 21041],), 'text': 'Josephe antwortete', 'tag': 'process'}, 'gold_standard': {'span': ([21023, 21041],), 'text': 'Josephe antwortete', 'tag': 'process'}}}, 3: {'annotations': {'Annotator1': {'span': (2590, 2687), 'text': 'Jeronimo wollte die Besinnung verlieren', 'tag': 'stative_event'}, 'Annotator2': {'span': (2590, 2687), 'text': 'Jeronimo wollte die Besinnung verlieren

In [ ]:
# with open("../data/261112_item_mapped_final_data.json", "w") as f:
#    json.dump(final_dict, f, indent=4)

## Finale Aufbereitung als Input für crowd-kit

In [288]:

rows = []

for title, matches in final_dict.items():

    for id, match_data in matches.items():
    
        for annotators, annotatorvalues in match_data["annotations"].items():
        
          

            rows.append({
                    "title": title,
                    "item":annotatorvalues["text"],
                    "task": id,
                    "worker": annotators,
                    "label": annotatorvalues["tag"]

            })

df = pd.DataFrame(rows)

In [289]:
display(df.head())

,title,item,task,worker,label
0,Das Erdbeben in Chili,"als ihr schon, auf Befehl des Erzbischofs, de...",1,Annotator1,process
1,Das Erdbeben in Chili,"als ihr schon, auf Befehl des Erzbischofs, der...",1,Annotator2,process
2,Das Erdbeben in Chili,"als ihr schon, auf Befehl des Erzbischofs, de...",1,gold_standard,process
3,Das Erdbeben in Chili,Josephe antwortete,2,Annotator1,process
4,Das Erdbeben in Chili,Josephe antwortete,2,Annotator2,process


In [ ]:
#df.to_csv("../data/261211_final_data.csv",index=False)

## Mappings prüfen

In [291]:
pd.set_option('display.max_rows', None)

### Haben gleiche Textsegmente unterschiedliche TASK-IDs?

In [292]:
same_items = (
    df.groupby(["title", "item"])
    .agg(
        tasks=("task", "unique"),
        workers=("worker", "unique"),
        n_tasks=("task", "nunique")
    )
    .reset_index()
)


diff_taskid_same_item = same_items[same_items["n_tasks"] > 1]

In [293]:
print("N items with different task ids: ",len(diff_taskid_same_item))

N items with different task ids:  184


In [294]:
display(diff_taskid_same_item)

,title,item,tasks,workers,n_tasks
67,Das Erdbeben in Chili,Don Fernando Ormez,"[274, 579]","[Annotator1, Annotator2, gold_standard]",2
170,Das Erdbeben in Chili,Ja,"[155, 439]","[Annotator1, Annotator2, gold_standard]",2
183,Das Erdbeben in Chili,Josephe antwortete,"[2, 414]","[Annotator1, Annotator2, gold_standard]",2
481,Das Erdbeben in Chili,den ihr alle kennt,"[137, 217]","[Annotator1, gold_standard]",2
482,Das Erdbeben in Chili,den ihr alle kennt,"[137, 217]",[Annotator2],2
760,Das Erdbeben in Chili,sie ist's,"[417, 629]","[Annotator1, Annotator2, gold_standard]",2
794,Das Erdbeben in Chili,steinigt sie,"[318, 412]","[Annotator1, Annotator2, gold_standard]",2
797,Das Erdbeben in Chili,und,"[9, 253, 442, 555]","[Annotator1, Annotator2, gold_standard]",4
911,Das Erdbeben in Chili,und fragte,"[92, 526, 534]","[Annotator1, Annotator2, gold_standard]",3
1369,Der blonde Eckbert,Ich wußte nicht,"[508, 802]","[Annotator1, Annotator2, gold_standard]",2


### Liegen für die selbe task-ID unterschiedliche items (Annotationstexte) vor?

In [295]:
# viele unterschiedliche Whitespaces vorhanden, daher entferenen bei doppelten und am Anfang und Ende
df["item"] = df["item"].str.replace(r"\s+", " ", regex=True).str.strip()

# Wie viele unterschiedliche Textsegmente gibt es pro Task?
# Group by title und task und dann check spalte item und count wie viele unique Einträge vorliegen 
span_comparison = (
    df.groupby(["title", "task"])["item"]
    .nunique()
    .reset_index(name="different_items")
)

# Nur Tasks anzeigen, bei denen die Textsegmente voneinander abweichen
same_id_diff_item = span_comparison[span_comparison["different_items"] > 1]
display(len(same_id_diff_item))

1117

In [297]:
#display(same_id_diff_item)